## מבנה נתונים רב-ממדי: דו-ממד ותלת-ממד

בשבוע 7 עבדנו עם מערך חד-ממדי: ציר אחד, בדרך כלל זמן. השבוע נרחיב לכמה צירים — וכל השבוע מונחה על ידי **ארבעה כללים**:

```{admonition} ארבעת הכללים לעבודה עם מערכים רב-ממדיים
:class: tip
הרעיון המארגן: **ציר הוא משמעות פיזיקלית, לא מספר**.

1. ‎`shape`‎ היא רשימת הצירים, ולכל מיקום יש שם — למשל (ריצה, זמן, קואורדינטה).
2. ‎`axis=k`‎ הוא הציר **שנעלם**. ‎`mean(axis=0)`‎ על מערך בצורה ‎`(5, 60)`‎ מחזיר ‎`(60,)`‎ — ממוצע על הריצות.
3. אפשר לבצע פעולת חשבון איבר-איבר בין שני מערכים אם, מימין לשמאל, בכל זוג צירים הגדלים שווים או שאחד מהם 1. אם לאחד המערכים יש פחות צירים, NumPy מוסיף לו צירים בגודל 1 משמאל.
4. ‎`None`‎ מוסיף ציר בגודל 1 — כלומר "המערך הזה קבוע לאורך הציר הזה".
```

בסעיף הזה נעסוק בכלל הראשון: לכל ציר יש **שם**, לא רק מספר. לשאלה "מה ה-shape המתקבל?" יש טעם רק אם אפשר לענות גם על "ומה זה אומר פיזיקלית?".

```{admonition} 🎥 כאן נכנס סרטון השבוע
:class: seealso

**"צירים ומערכים רב-ממדיים — ארבעת הכללים"** (כ-6–10 דקות, הקלטת מסך של המחברת עם קול). (הסרטון פרוש על סעיפים 9.1–9.6 כולם — כל אחד מהם מעמיק כלל אחר מתוך הארבעה. דוגמה נוספת בסרטון: סרטון וידאו עצמו הוא מערך תלת-ממדי (פריים, y, x).)

<!-- TODO (למפיק/ה): להחליף תא זה בהטמעת הסרטון בפועל, למשל:
<iframe width="100%" height="400" src="VIDEO_URL_HERE" title="שבוע 9 — צירים ומערכים רב-ממדיים — ארבעת הכללים" frameborder="0" allowfullscreen></iframe>
-->
```

### דוגמה: אותה זריקה במספר ריצות

בפועל לא זורקים פעם אחת: חוזרים על הניסוי כמה פעמים. נניח שחזרנו 5 פעמים על אותה זריקה ($\theta=45^\circ$), ומדדנו בכל פעם את מהירות השיגור, שיוצאת מעט שונה בכל ריצה. נחשב את הגובה $y$ באותן נקודות זמן בכל ריצה.

In [ ]:
import numpy as np

g = 9.8
theta = np.radians(45)
v0_nom = 20.0                       # nominal launch speed (m/s)
# measured launch speed of each run (m/s):
v0_runs = np.array([18.0, 19.5, 20.0, 20.5, 22.0])
n_runs, n_t = 5, 60
t_flight_nom = 2 * v0_nom * np.sin(theta) / g
t = np.linspace(0, t_flight_nom, n_t)   # the same time grid for all runs

# build Y run by run, with a loop (9.5 does it without one)
Y = np.zeros((n_runs, n_t))         # before the loop: an empty array
for i in range(n_runs):
    # every round: the heights of one run fill row i
    Y[i] = v0_runs[i] * np.sin(theta) * t - 0.5 * g * t**2

print("Y.shape =", Y.shape)         # after the loop: every row is filled

‎`Y.shape`‎ הוא ‎`(5, 60)`‎: **ציר 0** הוא הריצה (5 ריצות שונות), **ציר 1** הוא הזמן (60 נקודות). זו לא עובדה שרירותית על מספרים — ‎`Y[2]`‎ הוא **כל מסלול הריצה השלישית**, ו-‎`Y[:, 0]`‎ הוא **הגובה ההתחלתי בכל הריצות** (שכולן מתחילות מ-$y=0$).

### הרחבה לתלת-ממד: ציר קואורדינטה

אפשר להוסיף ציר שלישי: לא רק הגובה $y$, אלא גם המיקום האופקי $x$ — כך שכל ריצה, בכל זמן, מתוארת בשתי קואורדינטות.

In [ ]:
X = np.zeros((n_runs, n_t))
for i in range(n_runs):
    X[i] = v0_runs[i] * np.cos(theta) * t

XY = np.zeros((n_runs, n_t, 2))   # run, time, coordinate
XY[:, :, 0] = X                    # coordinate 0: x
XY[:, :, 1] = Y                    # coordinate 1: y
print("XY.shape =", XY.shape)

‎`XY.shape`‎ הוא ‎`(5, 60, 2)`‎: ריצה, זמן, קואורדינטה (0=$x$, 1=$y$). ‎`XY[2, 10, 1]`‎ הוא הגובה $y$ של הריצה השלישית, בנקודת הזמן ה-11. כל מספר במערך מאופיין בשלושה נתונים: הריצה, הזמן והקואורדינטה. זו המשמעות של "ציר בעל שם".

### נסו בעצמכם

בנו מערך ‎`V`‎ בצורה ‎`(n_runs, n_t)`‎ של **גודל המהירות** $|v(t)| = \sqrt{v_x^2+v_y^2}$ בכל ריצה ובכל זמן, בעזרת לולאה על הריצות (כמו ש-‎`Y`‎ נבנה למעלה). בדקו את התוצאה: ב-‎`t = 0`‎ גודל המהירות של כל ריצה שווה למהירות השיגור שלה.

In [ ]:
# V = np.zeros((n_runs, n_t))
# for i in range(n_runs):
#     vx_i = ...
#     vy_i = ...
#     V[i] = ...

`````{admonition} פתרון
:class: dropdown, tip
```python
V = np.zeros((n_runs, n_t))
for i in range(n_runs):
    vx_i = v0_runs[i] * np.cos(theta)
    vy_i = v0_runs[i] * np.sin(theta) - g * t
    V[i] = np.sqrt(vx_i**2 + vy_i**2)

print(V.shape)      # (5, 60)
print(V[:, 0])      # the speed of every run at t = 0: its launch speed
```
`````

### בדקו את עצמכם

השאלה הראשונה מתייחסת לקוד הבא (2 ריצות, 3 נקודות זמן):

```python
Y = np.array([[0.0, 1.0, 1.5],
              [0.0, 2.0, 3.0]])
print(Y[1])
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "כמה מספרים תדפיס שורת ההדפסה?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "3",
                "correct": true,
                "feedback": "נכון: אינדקס אחד בוחר ריצה שלמה, כלומר את הגובה בכל 3 נקודות הזמן שלה."
            },
            {
                "answer": "2",
                "correct": false,
                "feedback": "לא: 2 הוא מספר הריצות. גובה אחד מכל ריצה, בזמן אחד, מקבלים עם נקודתיים בציר הראשון."
            },
            {
                "answer": "1",
                "correct": false,
                "feedback": "לא: מספר בודד מקבלים רק עם שני אינדקסים, ריצה וזמן."
            },
            {
                "answer": "6",
                "correct": false,
                "feedback": "לא: 6 הוא מספר הערכים בכל המערך; אינדקס אחד בוחר רק ריצה אחת."
            }
        ]
    },
    {
        "question": "במערך XY מהדוגמה, מה מייצג הציר האחרון, שגודלו 2?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "הקואורדינטות x ו-y של כל נקודה",
                "correct": true,
                "feedback": "נכון."
            },
            {
                "answer": "שתי ריצות נוספות",
                "correct": false,
                "feedback": "לא — ציר הריצות הוא הראשון, בגודל 5."
            },
            {
                "answer": "שני מכשירי מדידה שונים",
                "correct": false,
                "feedback": "לא בהכרח נכון בהקשר הזה — כאן זה x ו-y."
            },
            {
                "answer": "שני סוגי חיישנים שונים",
                "correct": false,
                "feedback": "לא בהכרח נכון בהקשר הזה — כאן הגודל 2 מייצג את x ו-y של אותה נקודה, לא חיישנים שונים."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

``````{admonition} תרגול נוסף
:class: dropdown, note
בנו מערך ‎`A`‎ בצורה ‎`(3, n_t)`‎ של **התאוצה** האנכית ($-g$, קבועה בכל ריצה ובכל זמן) עבור 3 ריצות, בעזרת לולאה. ודאו ש-‎`A.shape`‎ הוא ‎`(3, n_t)`‎.

```python
A = np.zeros((3, n_t))
# ...
```

`````{admonition} פתרון
:class: dropdown, tip
```python
A = np.zeros((3, n_t))
for i in range(3):
    A[i] = -g   # the same acceleration at every time

print(A.shape)
```
`````
``````